# Saber Denoising Process Visualization (GSM8K)

在 GSM8K 上跑 50 个 task，使用 Saber (AADU + BERM cross_step + Expand) 生成，
收集每步解码快照，生成一个 HTML 文件供离线浏览。

**配置**: `saber_expand=True, berm_mode=cross_step, saber_global_aadu=True, saber_mtr=0.8, saber_n=4, saber_mu=8, block_length=32`

## 1. 环境设置

In [ ]:
import os, sys, gc
import torch
import torch.nn.functional as F
import numpy as np

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
os.chdir(os.path.join(NOTEBOOK_DIR, 'llada'))
print(f'Working dir: {os.getcwd()}')

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

## 2. 模型 & 数据加载

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM
from datasets import load_dataset

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True
model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'Model loaded. GSM8K test: {len(gsm8k)} samples')

In [ ]:
FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(formatted)['input_ids']
    return torch.tensor(ids, dtype=torch.long, device='cuda').unsqueeze(0)


import re

def extract_answer(text: str) -> str | None:
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return m.group(1).replace(',', '').strip() if m else None


LIMIT = 50
prompts = [build_prompt(gsm8k[i]['question']) for i in range(LIMIT)]
questions = [gsm8k[i]['question'] for i in range(LIMIT)]
ref_answers = [gsm8k[i]['answer'] for i in range(LIMIT)]
print(f'Built {len(prompts)} prompts, first length: {prompts[0].shape[1]} tokens')

## 3. Saber 生成 + 收集 denoising 数据

核心函数：`generate_with_collection_saber` — 完整复刻 `generate_with_expand_saber` 的逻辑，
同时在每步收集快照（x / x0 / mask / confidence / block_range），
输出格式与 `viz_static.process_sample` 兼容。

In [ ]:
from generate import add_gumbel_noise


def _saber_select(confidence, mask, conf_sum, conf_count, saber_n):
    """AADU: pick tokens to unmask.

    Returns:
        sel: selected indices
        tau: adaptive threshold
        forced_sel: subset of sel forced by n-floor (not from conf>=tau)
    """
    n_masked = int(mask.sum().item())
    if n_masked == 0:
        empty = torch.empty(0, dtype=torch.long, device=confidence.device)
        return empty, 0.0, empty

    if conf_count > 0:
        tau = (conf_sum / conf_count).item()
    else:
        tau = confidence[mask].max().item()

    cand = torch.where(confidence >= tau)[0]
    sel = cand
    forced_sel = torch.empty(0, dtype=torch.long, device=confidence.device)
    if sel.numel() < saber_n:
        k = min(saber_n, n_masked)
        _, sel = torch.topk(confidence, k=k)
        # forced-by-floor = selected but not in conf>=tau candidate set
        if cand.numel() > 0:
            forced_sel = sel[~torch.isin(sel, cand)]
        else:
            forced_sel = sel

    return sel, tau, forced_sel


def _saber_berm_cross_step(x_block, full_conf, last_conf, unmask_time_conf,
                            conf_sum, conf_count, n_unmask, saber_n, saber_mu, mask_id):
    """BERM: remask suspicious tokens based on cross-step confidence delta.

    Returns:
        remasked_count, conf_sum, conf_count, rem_idx_local
    """
    still_masked = (x_block == mask_id)
    delta = full_conf - last_conf
    delta_rem = delta.clone()
    delta_rem[still_masked] = float('inf')

    n_eligible = int((~still_masked).sum().item())
    mu_t = max(saber_n // 2, (n_unmask + saber_mu - 1) // saber_mu)
    mu_t = min(mu_t, max(0, n_unmask - 1))
    mu_t = min(mu_t, n_eligible)
    if mu_t <= 0:
        return 0, conf_sum, conf_count, []

    _, rem_idx = torch.topk(delta_rem, k=mu_t, largest=False)
    remasked = 0
    rem_idx_local = []
    for ri in rem_idx:
        ri_v = ri.item()
        if x_block[ri_v] != mask_id:
            x_block[ri_v] = mask_id
            conf_sum -= unmask_time_conf[ri_v]
            conf_count -= 1
            unmask_time_conf[ri_v] = 0.0
            remasked += 1
            rem_idx_local.append(ri_v)
    return remasked, conf_sum, conf_count, rem_idx_local


@torch.no_grad()
def generate_with_collection_saber(
    model, prompt, steps=256, gen_length=256, block_length=32,
    temperature=0., mask_id=126336,
    mid_trigger_ratio=0.8, saber_n=4, saber_mu=8, global_aadu=True,
    verbose=False, sample_idx=None,
):
    """
    Saber (AADU + BERM cross_step + Expand) generation with snapshot collection.
    Output format compatible with viz_static.process_sample.
    """
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    seq_len = Lp + gen_length
    device = model.device
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    trigger_thresh = int(block_length * mid_trigger_ratio)
    NEG_INF = torch.tensor(float('-inf'), device=device, dtype=torch.float64)

    x = torch.full((B, seq_len), mask_id, dtype=torch.long, device=device)
    x[:, :Lp] = prompt
    nfe = 0

    g_conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
    g_conf_count = torch.zeros(B, device=device, dtype=torch.long)
    pconf = torch.zeros(B, seq_len, dtype=torch.float64, device=device)

    history = []
    global_step = 0

    if verbose:
        sid = '?' if sample_idx is None else sample_idx + 1
        print(f'[diag] sample {sid}: prompt_len={Lp}, gen_length={gen_length}, block_length={block_length}, num_blocks={num_blocks}')

    def _snap(step_i, nb, s, e, x0_full_seq, forced_floor_pos=None, berm_remask_pos=None):
        mask_now = (x == mask_id)
        return {
            'step': step_i,
            'block_id': nb,
            'x': x.cpu().numpy().copy(),
            'x0': x0_full_seq.cpu().numpy().copy(),
            'mask': mask_now.cpu().numpy().copy(),
            'confidence': pconf.cpu().numpy().copy(),
            'current_block_range': (s, e),
            'forced_floor_pos': forced_floor_pos or [[] for _ in range(B)],
            'berm_remask_pos': berm_remask_pos or [[] for _ in range(B)],
        }

    nb = 0
    while nb < num_blocks:
        s = Lp + nb * block_length
        e = s + block_length
        block_steps = []
        if verbose:
            print(f'[diag] block {nb}/{num_blocks - 1}: token_range=[{s}, {e}) start')

        if global_aadu:
            conf_sum = g_conf_sum.clone()
            conf_count = g_conf_count.clone()
        else:
            conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
            conf_count = torch.zeros(B, device=device, dtype=torch.long)

        BL = e - s
        last_conf = torch.zeros(B, BL, device=device, dtype=torch.float64)
        unmask_tc = torch.zeros(B, BL, device=device, dtype=torch.float64)

        # Phase 1: warm-up
        out = model(x, use_cache=True)
        past_kv = out.past_key_values
        nfe += 1
        blk_logits = out.logits[:, s:e, :]
        del out

        x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
        p = F.softmax(blk_logits.to(torch.float64), dim=-1)
        x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
        blk_mask = (x[:, s:e] == mask_id)

        forced_floor_pos = [[] for _ in range(B)]
        for j in range(B):
            conf_j = torch.where(blk_mask[j], x0_p[j], NEG_INF)
            sel, _, forced_sel = _saber_select(conf_j, blk_mask[j], conf_sum[j], conf_count[j], saber_n)
            if sel.numel() == 0:
                continue
            x[j, s + sel] = x0[j, sel]
            sc = x0_p[j, sel]
            conf_sum[j] += sc.sum()
            conf_count[j] += sel.numel()
            unmask_tc[j, sel] = sc
            if forced_sel.numel() > 0:
                forced_floor_pos[j].extend([int(s + k.item()) for k in forced_sel])

        last_conf = x0_p.clone()
        pconf[:, s:e] = x0_p

        x0_snap = x.clone()
        x0_snap[:, s:e] = x0
        block_steps.append(_snap(global_step, nb, s, e, x0_snap, forced_floor_pos=forced_floor_pos))
        if verbose:
            decoded_now = int((x[:, s:e] != mask_id).sum().item())
            print(f'[diag] block {nb}: warm-up done, nfe={nfe}, decoded_in_window={decoded_now}/{B * (e - s)}')
        global_step += 1

        # Phase 2: refinement with expand
        rp = torch.zeros(B, seq_len, dtype=torch.bool, device=device)
        rp[:, s:e] = True
        watching_nb = nb
        blocks_consumed = 1

        for _step in range(steps):
            if not (x[:, s:e] == mask_id).any():
                break

            # Expand check
            can_expand = (watching_nb + 1 < num_blocks)
            if can_expand:
                wb_s = Lp + watching_nb * block_length
                wb_e = wb_s + block_length
                remaining = int((x[:, wb_s:wb_e] == mask_id).sum(dim=1).max().item())
                if remaining <= trigger_thresh:
                    next_nb = watching_nb + 1
                    e_new = min(Lp + (next_nb + 1) * block_length, seq_len)
                    if verbose:
                        print(f'[diag] block {nb}: expand triggered at step={global_step}, watching_nb={watching_nb}, remaining={remaining}, new_end={e_new}')
                    e = e_new
                    BL = e - s

                    if not global_aadu:
                        conf_sum = torch.zeros(B, device=device, dtype=torch.float64)
                        conf_count = torch.zeros(B, device=device, dtype=torch.long)

                    out = model(x, use_cache=True)
                    past_kv = out.past_key_values
                    nfe += 1
                    blk_logits = out.logits[:, s:e, :]
                    del out

                    x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
                    p = F.softmax(blk_logits.to(torch.float64), dim=-1)
                    x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
                    blk_mask = (x[:, s:e] == mask_id)

                    forced_floor_pos = [[] for _ in range(B)]
                    for j in range(B):
                        conf_j = torch.where(blk_mask[j], x0_p[j], NEG_INF)
                        sel, _, forced_sel = _saber_select(conf_j, blk_mask[j], conf_sum[j], conf_count[j], saber_n)
                        if sel.numel() == 0:
                            continue
                        x[j, s + sel] = x0[j, sel]
                        sc = x0_p[j, sel]
                        conf_sum[j] += sc.sum()
                        conf_count[j] += sel.numel()
                        if forced_sel.numel() > 0:
                            forced_floor_pos[j].extend([int(s + k.item()) for k in forced_sel])

                    last_conf = torch.zeros(B, BL, device=device, dtype=torch.float64)
                    unmask_tc = torch.zeros(B, BL, device=device, dtype=torch.float64)
                    last_conf[:, :x0_p.shape[1]] = x0_p
                    for j in range(B):
                        unmasked_j = (x[j, s:e] != mask_id)
                        unmask_tc[j, :unmasked_j.shape[0]][unmasked_j] = x0_p[j][unmasked_j]

                    rp = torch.zeros(B, seq_len, dtype=torch.bool, device=device)
                    rp[:, s:e] = True
                    blocks_consumed += 1
                    watching_nb = next_nb

                    pconf[:, s:e] = x0_p
                    x0_snap = x.clone()
                    x0_snap[:, s:e] = x0
                    block_steps.append(_snap(global_step, nb, s, e, x0_snap, forced_floor_pos=forced_floor_pos))
                    global_step += 1
                    continue

            # Normal refinement step
            blk_logits = model(
                x[:, s:e], past_key_values=past_kv,
                use_cache=True, replace_position=rp,
            ).logits
            nfe += 1

            x0 = torch.argmax(add_gumbel_noise(blk_logits, temperature), dim=-1)
            p = F.softmax(blk_logits.to(torch.float64), dim=-1)
            x0_p = torch.gather(p, -1, x0.unsqueeze(-1)).squeeze(-1)
            blk_mask = (x[:, s:e] == mask_id)
            full_conf = x0_p.clone()

            forced_floor_pos = [[] for _ in range(B)]
            berm_remask_pos = [[] for _ in range(B)]
            for j in range(B):
                conf_j = torch.where(blk_mask[j], x0_p[j], NEG_INF)
                sel, _, forced_sel = _saber_select(conf_j, blk_mask[j], conf_sum[j], conf_count[j], saber_n)
                n_unmask = sel.numel()
                if n_unmask == 0:
                    continue
                x[j, s + sel] = x0[j, sel]
                sc = x0_p[j, sel]
                conf_sum[j] += sc.sum()
                conf_count[j] += sel.numel()
                unmask_tc[j, sel] = sc

                if forced_sel.numel() > 0:
                    forced_floor_pos[j].extend([int(s + k.item()) for k in forced_sel])

                _, conf_sum[j], conf_count[j], rem_idx_local = _saber_berm_cross_step(
                    x[j, s:e], full_conf[j], last_conf[j], unmask_tc[j],
                    conf_sum[j], conf_count[j], n_unmask, saber_n, saber_mu, mask_id,
                )
                if rem_idx_local:
                    berm_remask_pos[j].extend([int(s + k) for k in rem_idx_local])

            last_conf = full_conf.clone()
            pconf[:, s:e] = torch.where(blk_mask, x0_p, pconf[:, s:e])

            x0_snap = x.clone()
            x0_snap[:, s:e] = torch.where(blk_mask, x0, x[:, s:e])
            block_steps.append(_snap(
                global_step, nb, s, e, x0_snap,
                forced_floor_pos=forced_floor_pos,
                berm_remask_pos=berm_remask_pos,
            ))
            if verbose and ((_step < 3) or ((_step + 1) % 20 == 0)):
                decoded_now = int((x[:, s:e] != mask_id).sum().item())
                remasked_now = sum(len(v) for v in berm_remask_pos)
                forced_now = sum(len(v) for v in forced_floor_pos)
                print(f'[diag] block {nb}: refine_step={_step + 1}, nfe={nfe}, decoded_in_window={decoded_now}/{B * (e - s)}, forced={forced_now}, berm_remask={remasked_now}')
            global_step += 1

        if global_aadu:
            g_conf_sum = conf_sum.clone()
            g_conf_count = conf_count.clone()

        history.append({'block_id': nb, 'steps': block_steps})
        if verbose:
            print(f'[diag] block {nb}: done, blocks_consumed={blocks_consumed}, total_nfe={nfe}, collected_steps={len(block_steps)}')
        nb += blocks_consumed

    if verbose:
        print(f'[diag] sample generation finished: total_nfe={nfe}, total_history_blocks={len(history)}')
    return x, nfe, history


def process_sample_saber(
    sample_blocks: list,
    tokenizer,
    gen_length: int,
    block_length: int,
    batch_idx: int = 0,
    question: str | None = None,
    gen_answer: str | None = None,
    ref_answer: str | None = None,
) -> dict:
    """Reuse viz_static.process_sample and only add Saber-specific tags.

    This keeps behavior/UI consistent with viz_denoising.ipynb while attaching:
      - f4: forced by saber_n floor
      - brm: remasked by BERM
    """
    from viz_static import process_sample as _base_process_sample

    result = _base_process_sample(
        sample_blocks,
        tokenizer,
        batch_idx=batch_idx,
        question=question,
        gen_answer=gen_answer,
        ref_answer=ref_answer,
    )

    # Keep block display stable: physical ranges, not dynamic expanded windows.
    gen_start = int(result['gen_start'])
    num_blocks = gen_length // block_length
    result['num_blocks'] = num_blocks
    result['block_ranges'] = [
        [gen_start + i * block_length, gen_start + (i + 1) * block_length]
        for i in range(num_blocks)
    ]

    # Attach Saber tags per step/token while preserving base output structure.
    step_idx = 0
    for block in sample_blocks:
        for sd in block['steps']:
            ff_raw = sd.get('forced_floor_pos')
            br_raw = sd.get('berm_remask_pos')

            if isinstance(ff_raw, list) and len(ff_raw) > batch_idx and isinstance(ff_raw[batch_idx], list):
                ff_set = set(int(p) for p in ff_raw[batch_idx])
            else:
                ff_set = set()

            if isinstance(br_raw, list) and len(br_raw) > batch_idx and isinstance(br_raw[batch_idx], list):
                br_set = set(int(p) for p in br_raw[batch_idx])
            else:
                br_set = set()

            for tk in result['steps'][step_idx]['tk']:
                p = int(tk.get('p', -1))
                tk['f4'] = (p in ff_set)
                tk['brm'] = (p in br_set)

            step_idx += 1

    return result


print('generate_with_collection_saber + process_sample_saber defined.')

## 4. 跑 50 个 task + 收集 denoising 数据

In [ ]:
import time

GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
SABER_N = 4
SABER_MU = 8
SABER_MTR = 0.8

all_samples_json = []

t0 = time.time()
for i in range(LIMIT):
    prompt = prompts[i]
    sample_t0 = time.time()
    print(f'[diag] start sample {i+1}/{LIMIT}')

    x, nfe, history = generate_with_collection_saber(
        model, prompt,
        steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
        temperature=0.0, mask_id=MASK_ID,
        mid_trigger_ratio=SABER_MTR,
        saber_n=SABER_N, saber_mu=SABER_MU, global_aadu=True,
        verbose=(i == 0), sample_idx=i,
    )
    print(f'[diag] sample {i+1}: generation returned in {time.time()-sample_t0:.1f}s (nfe={nfe}, history_blocks={len(history)})')

    gen_text = tokenizer.decode(x[0, prompt.shape[1]:], skip_special_tokens=True)
    for stop in ['Question:', '\n\nQuestion']:
        if stop in gen_text:
            gen_text = gen_text.split(stop)[0]
    gen_ans = extract_answer(gen_text)
    ref_ans = extract_answer(ref_answers[i])
    correct = gen_ans is not None and ref_ans is not None and gen_ans == ref_ans

    tag = chr(0x2713) if correct else chr(0x2717) + ' (ref: ' + str(ref_ans) + ')'
    sample_json = process_sample_saber(
        history, tokenizer,
        gen_length=GEN_LENGTH,
        block_length=BLOCK_LENGTH,
        question=questions[i],
        gen_answer=f'{gen_ans} {tag}',
        ref_answer=ref_ans,
    )
    print(f'[diag] sample {i+1}: process_sample_saber done in {time.time()-sample_t0:.1f}s')
    all_samples_json.append(sample_json)

    elapsed = time.time() - t0
    mark = chr(0x2713) if correct else chr(0x2717)
    print(f'[{i+1}/{LIMIT}] NFE={nfe:3d} ans={gen_ans} {mark} ({elapsed:.1f}s)')

correct_count = sum(1 for s in all_samples_json if chr(0x2713) in s.get('gen_answer', ''))
print(f'\nDone! {correct_count}/{LIMIT} correct. Total time: {time.time()-t0:.1f}s')

## 5. 生成 HTML

In [ ]:
from viz_static import generate_multi_html

html = generate_multi_html(
    all_samples_json,
    title=f'GSM8K Saber Denoising (n={SABER_N}, mu={SABER_MU}, bl={BLOCK_LENGTH}) — {LIMIT} samples',
)

output_path = os.path.join(NOTEBOOK_DIR, 'viz_gsm8k_saber_50.html')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html)

size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f'HTML saved: {output_path}')
print(f'  Size: {size_mb:.1f} MB')
print(f'  下载到本地用浏览器打开即可浏览全部 {LIMIT} 个 sample 的 Saber denoising 过程')

In [ ]:
# (可选) 在 notebook 内预览
from IPython.display import IFrame
IFrame(src=output_path, width='100%', height=700)